# 🌾 Crop Price Predictor — Exploratory Data Analysis

This notebook explores the Kaggle Indian Mandi price dataset before training the ML model.

**Run this notebook from the `crop-price-predictor/` root directory.**

In [ ]:
import sys
import os

# Make sure the ml/ module is importable
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load Raw Data

In [ ]:
DATA_PATH = '../data/daily_price.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Column names and data types
print('Columns:', list(df_raw.columns))
print()
print(df_raw.dtypes)

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

In [ ]:
# Duplicate rows
print(f'Duplicate rows: {df_raw.duplicated().sum():,}')

## 2. Load Cleaned Data

In [ ]:
from ml.preprocess import load_clean_data

df = load_clean_data(DATA_PATH)
print(f'Clean shape: {df.shape}')
df.head()

## 3. Unique Values Overview

In [ ]:
print(f'Unique commodities : {df["commodity"].nunique()}')
print(f'Unique states      : {df["state"].nunique()}')
print(f'Unique districts   : {df["district"].nunique()}')
print(f'Unique markets     : {df["market"].nunique()}')
print(f'Date range         : {df["date"].min().date()} → {df["date"].max().date()}')

print('\nTop 20 commodities by record count:')
print(df['commodity'].value_counts().head(20))

## 4. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes, ['min_price', 'max_price', 'modal_price'],
                           ['Min Price', 'Max Price', 'Modal Price']):
    data = df[col].clip(upper=df[col].quantile(0.99))  # Remove extreme outliers for viz
    ax.hist(data, bins=60, color='#3b82d4', edgecolor='white', alpha=0.8)
    ax.set_title(f'{title} Distribution')
    ax.set_xlabel('Price (INR/quintal)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Modal Price Over Time (Top Commodity)

In [ ]:
top_commodity = df['commodity'].value_counts().index[0]
df_top = df[df['commodity'] == top_commodity].copy()

# Monthly average
df_top_monthly = (
    df_top.groupby(df_top['date'].dt.to_period('M'))['modal_price']
    .mean()
    .reset_index()
)
df_top_monthly['date'] = df_top_monthly['date'].dt.to_timestamp()

fig = px.line(
    df_top_monthly, x='date', y='modal_price',
    title=f'Monthly Average Modal Price — {top_commodity}',
    labels={'modal_price': 'Modal Price (INR/quintal)', 'date': 'Month'},
)
fig.show()

## 6. Top States by Records

In [ ]:
top_states = df['state'].value_counts().head(15)
fig = px.bar(x=top_states.index, y=top_states.values,
             labels={'x': 'State', 'y': 'Number of Records'},
             title='Top 15 States by Number of Price Records')
fig.show()

## 7. Correlation: Min / Max / Modal Price

In [ ]:
corr = df[['min_price', 'max_price', 'modal_price']].corr()
sns.heatmap(corr, annot=True, cmap='YlGn', fmt='.3f')
plt.title('Price Correlation Matrix')
plt.show()

## 8. Feature Importance (after training)

Run `python ml/train.py` first, then execute the cell below.

In [ ]:
import joblib, json
import numpy as np

pipeline = joblib.load('../ml/saved_model.pkl')
with open('../ml/metrics.json') as f:
    metrics = json.load(f)

cat_features = metrics['features_categorical']
num_features = metrics['features_numerical']

# Get feature names after OHE
ohe = pipeline.named_steps['preprocessor'].transformers_[0][1]
ohe_feature_names = ohe.get_feature_names_out(cat_features).tolist()
all_feature_names = ohe_feature_names + num_features

importances = pipeline.named_steps['regressor'].feature_importances_
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

fig = px.bar(
    x=[all_feature_names[i] for i in top_idx],
    y=[importances[i] for i in top_idx],
    title=f'Top {top_n} Feature Importances (RandomForest)',
    labels={'x': 'Feature', 'y': 'Importance'},
)
fig.update_xaxes(tickangle=45)
fig.show()

print(f"\nModel Metrics:")
print(f"  MAE  = {metrics['MAE']:,.2f}")
print(f"  RMSE = {metrics['RMSE']:,.2f}")
print(f"  R²   = {metrics['R2']:.4f}")